In [0]:
%sql
select
  encounter_class,
  quarter(start),
  count(*)
    / (
      select
        count(*)
      from
        medical_pipeline.silver.encounters_silver
    )
    * 100 as percentage
from
  medical_pipeline.silver.encounters_silver
group by
  encounter_class,
  quarter(start)
order by
  encounter_class,
  quarter(start);

In [0]:
%sql
with table1 as (
  select
    *,
    TIMESTAMPDIFF(HOUR, start, stop) as time_diff,
    month(start) as month
  from
    medical_pipeline.silver.encounters_silver
),
table2 as (
  select
    `month`,
    count(*) as more_than_24
  from
    table1
  where time_diff > 24
  group by
    `month`

    
  order by
    `month`
),
table3 as (
  select
    `month`,
    count(*) as less_than_24
  from
    table1
    where time_diff < 24
  group by
    `month`

  order by
    `month`
)
select
  t2.`month`,
  more_than_24,
  less_than_24,
  (more_than_24 / less_than_24)*100 as `over_24/under_24`
from
    table2 t2  
    join table3 t3
      on t2.`month` = t3.`month`
order by
  `month`;

In [0]:
%sql
with table1 as (
  select
    * ,  month(start) as `month`
  from
    medical_pipeline.silver.encounters_silver
) , table2 as (
  select
    `month`, 
    count(*) as total,
    sum(case when payer_coverage = 0 then 1 else 0 end) as zero_payer_coverage
    from table1 
    group by `month`
) select `month`, total, zero_payer_coverage, (zero_payer_coverage/total)*100 as zero_payer_coverage_pct ,((total-zero_payer_coverage)/total)*100 as non_zero_payer_coverage_percentage from table2;